In [26]:
from pathlib import Path
from zipfile import ZipFile

from open_video_summary.utils.config import DataPaths
from open_video_summary.parsers.video import VideoLoader, VideoDumper
from open_video_summary.core.segmenter.video_segmenter import WordVideoSegmenter

SyntaxError: invalid syntax (video_segmenter.py, line 74)

# Unzipping sample dataset video files

In [2]:
zip_path = (Path(DataPaths.RAW_PATH) / "bebe_real.zip").as_posix()
extract_path = (Path(DataPaths.RAW_PATH)).as_posix()

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

processed_data = Path(DataPaths.RAW_PATH) / "bebe_real" / "bebe_real.json"
processed_data.rename(Path(DataPaths.PROCESSED_PATH) / "bebe_real.json")

PosixPath('/home/leo/phd/repos/open-video-summary/data/processed/bebe_real.json')

# Loading data, segmenting videos, and saving output file

In [ ]:

datasets = [
    "bebe_real",
    "bolsonaro_trump",
    "brumadinho",
    "catedral_notre_dame",
    "google_huawei",
]
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (Path(DataPaths.PROCESSED_PATH) / f"{dataset}.json").as_posix()

    video_segmenter = WordVideoSegmenter(
        whisper_model="medium",
        min_segment_length=5,
        max_segment_length=120,
    )

    videos = [
        raw_video for raw_video in VideoLoader.load_videos_from_directory(dataset_videos_path)
    ]


    videos = video_segmenter.create_videos_segments(videos)
    VideoDumper.dump_videos_to_json(videos, dataset_json_path)

[00:00.720 --> 00:00.880] O
[00:00.880 --> 00:01.200] príncipe
[00:01.200 --> 00:01.660] Harry
[00:01.660 --> 00:01.880] e
[00:01.880 --> 00:02.000] a
[00:02.000 --> 00:02.520] doquesa
[00:02.520 --> 00:02.840] Meghan
[00:02.840 --> 00:03.400] Markle
[00:03.400 --> 00:04.160] apresentaram
[00:04.160 --> 00:04.500] hoje
[00:04.500 --> 00:04.660] o
[00:04.660 --> 00:04.900] filho
[00:04.900 --> 00:05.780] recém-nascido.
[00:07.560 --> 00:07.740] A
[00:07.740 --> 00:07.900] primeira
[00:07.900 --> 00:08.360] aparição
[00:08.360 --> 00:08.520] de
[00:08.520 --> 00:08.800] Harry
[00:08.800 --> 00:09.000] foi
[00:09.000 --> 00:09.180] aos
[00:09.180 --> 00:09.420] olhos
[00:09.420 --> 00:09.720] de
[00:09.720 --> 00:09.960] todos.
[00:10.420 --> 00:10.620] Uma
[00:10.620 --> 00:11.060] multidão
[00:11.060 --> 00:11.220] se
[00:11.220 --> 00:11.680] acutuvelou
[00:11.680 --> 00:11.820] em
[00:11.820 --> 00:11.960] frente
[00:11.960 --> 00:12.100] ao
[00:12.100 --> 00:12.420] hospital.
[00:14.

In [27]:

datasets = [
    "bebe_real",
    "bolsonaro_trump",
    "brumadinho",
    "catedral_notre_dame",
    "google_huawei",
]

data = []
for dataset in datasets:
    dataset_videos_path = (Path(DataPaths.RAW_PATH) / dataset).as_posix()
    dataset_json_path = (Path(DataPaths.PROCESSED_PATH) / f"{dataset}.json").as_posix()

    videos = VideoLoader.load_videos_from_json(dataset_json_path)
    for video in videos:
        for i, seg in enumerate(video.segments):
            data.append((dataset, video.name, seg.content, seg.start, seg.end))
    VideoDumper.dump_videos_to_json(videos, dataset_json_path)

In [21]:
import pandas as pd

df = pd.DataFrame(data, columns=["dataset", "video_name", "segment", "start_time", "end_time"])
df.start_time = df.start_time.astype(str).replace("\.", ",", regex=True)
df.end_time = df.end_time.astype(str).replace("\.", ",", regex=True)
df

,dataset,video_name,segment,start_time,end_time
0,bebe_real,jornal_nacional,O príncipe Harry e a doquesa Meghan Markle apr...,"0,72","41,48"
1,bebe_real,jornal_nacional,O casal também quebrou hoje outro mistério e r...,"43,0","52,24"
2,bebe_real,jornal_nacional,Elizabeth Alexandra Mary conheceu hoje o bisne...,"52,72","67,48"
3,bebe_real,jornal_da_band,Harry e a duquesa Meghan Markle apareceram em ...,"0,0","6,06"
4,bebe_real,jornal_da_band,O bebê real recebeu o nome de Archie Harrison....,"6,28","24,72"
...,...,...,...,...,...
117,google_huawei,jornal_nacional,Sistema operacional é o que faz o smartphone f...,"7,38","22,24"
118,google_huawei,jornal_nacional,"Durante muito tempo, eles cresceram se ajudand...","22,62","93,9"
119,google_huawei,jornal_nacional,A nova tecnologia 5G promete revolucionar a in...,"94,46","103,22"
120,google_huawei,jornal_da_band,O Google decidiu não atualizar mais os aplicat...,"0,08","43,64"


In [22]:
df.to_csv("file.csv", encoding="latin1", index=False)

In [ ]:
import ollama
from ast import literal_eval
from open_video_summary.core.segmenter.prompts import VideoSegmenterPrompts

all_topics = [
    "Apresentação do bebê real",
    "Estilo discreto do casal",
    "Nome do bebê Archie Harrison Mountbatten-Windsor",
    "Reação da família real ao nascimento",
    "Apresentação pública do bebê real Archie Harrison",
    "Detalhes sobre o temperamento e características do bebê",
    "Linha de sucessão e histórico familiar do bebê",
    "Apresentação do bebê real",
    "Nome e nacionalidade do bebê",
    "Reação da família real",
    "Expectativas sobre o título do bebê",
    "Apresentação do bebê real",
    "Linha de sucessão",
    "Reação da família real",
    "Nome e nacionalidade do bebê",
    "Temperamento",
]

prompt = VideoSegmenterPrompts.get_global_topics.format(
    topics_collection=all_topics
)

response = ollama.generate(model="gemma2", prompt=prompt, options={"format": "json", "temperature": .2})
response = response.get("response")

global_topics = {i: topic for i, topic in enumerate(literal_eval(response))}
global_topics

["Apresentação do bebê real", "Estilo discreto do casal", "Nome e nacionalidade do bebê", "Reação da família real", "Linha de sucessão", "Detalhes sobre o temperamento e características do bebê"] 



In [18]:
from open_video_summary.core.segmenter.prompts import VideoSegmenterPrompts
global_topics

{0: 'Apresentação do bebê real',
 1: 'Estilo discreto do casal',
 2: 'Nome e nacionalidade do bebê',
 3: 'Reação da família real',
 4: 'Linha de sucessão',
 5: 'Detalhes sobre o temperamento e características do bebê'}

In [ ]:

prompt = VideoSegmenterPrompts.classify_subtopic.format(
    content="O príncipe Harry e a doquesa Meghan Markle apresentaram hoje o filho recém-nascido. A primeira aparição de Harry foi aos olhos de todos. Uma multidão se acutuvelou em frente ao hospital. O primeiro vislumbre do filho do príncipe foi só para um seleto grupo da imprensa. E junto com o novo bebê real, o casal apresentou um estilo discreto. O menino estava embrulhado em carinhos. Meghan Markle falou da magia que é virar mãe, sobretudo por ter dois dos melhores caras do mundo. O bebê vem fazendo por merecer. Temperamento calmo e doce, Harry diz que nem sabe a quem ele puxou.",
    topics=global_topics
)
response = ollama.generate(model="gemma2", prompt=prompt, options={"format": "json", "temperature": .2})
response = response.get("response")
print(response)

```json
{
"0": "Apresentação do bebê real"
}
``` 


The document primarily focuses on the public presentation of the newborn baby, including details about the event and the reactions to it.  



In [35]:
from re import search, DOTALL
result = search("(\{.*?\})", response, flags=DOTALL)
if result is not None:
    result = result.group(0)
segment_topic = literal_eval(result)
topic_id, _ = segment_topic.popitem()
global_topics[topic_id]

KeyError: '0'